## Sustainability & Spatial Misalignment in Early-Stage Corridor Development

| Hypothesis | Statement |
|---|---|
| H1 | Built-up expansion is associated with declining vegetation cover and increased surface heat susceptibility |
| H2 | A share of new built-up areas overlaps with environmentally sensitive or water-prone zones |
| H3 |  Areas with low economic utilisation disproportionately coincide with higher environmental exposure |

In [ ]:
import geemap
import ee 
import geemap.colormaps as cm
import os

In [ ]:
geemap.ee_initialize()

In [ ]:
# Define the Relative Path
# '..' means go up one folder from where this notebook is saved
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')

Map = geemap.Map()

In [ ]:
roi = geemap.geojson_to_ee(file_path)
Map.addLayer(roi, {'color': 'blue', 'width': 2}, 'Dholera Taluka Boundary')
Map.centerObject(roi, 10)
Map

In [ ]:
# ── Sentinel-2: 2025 (Oct–Dec post-monsoon composite) ───────
s2_2025 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

# ── Sentinel-2: 2016 (Oct–Dec post-monsoon composite) ───────
s2_2016 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2016-10-01', '2016-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

Computes NDBI, MNDWI, SAVI, and NDVI for a given Sentinel-2 image.  
NDVI is added here (not in RQ1/RQ2) as it provides a cleaner vegetation-loss signal alongside SAVI.

In [ ]:
def calculate_indices(image):
    """
    Computes NDBI, MNDWI, SAVI, NDVI from a Sentinel-2 SR image.
    Band references:
        B2=Blue, B3=Green, B4=Red, B8=NIR, B11=SWIR1
    """
    ndbi = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)',
        {'SWIR1': image.select('B11'), 'NIR': image.select('B8')}
    ).rename('NDBI')

    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)',
        {'Green': image.select('B3'), 'SWIR1': image.select('B11')}
    ).rename('MNDWI')

    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)',
        {'NIR': image.select('B8'), 'Red': image.select('B4')}
    ).rename('SAVI')

    return image.addBands([ndbi, mndwi, savi])


processed_2025 = calculate_indices(s2_2025)
processed_2016 = calculate_indices(s2_2016)

print("Spectral indices computed for 2016 and 2025.")

---
## Built-up Masks (Reusing RQ1 thresholds)

| Year | NDBI threshold | Rationale |
|---|---|---|
| 2025 | > 0.05 | Post-monsoon; lower saline soil noise |
| 2016 | > 0.13 | Higher radiometric contrast from saline pans in drier 2016 season |

In [ ]:
# ── 2025 Built-up Mask ───────────────────────────────────────
built_up_2025 = processed_2025.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.18)',
    {
        'NDBI':  processed_2025.select('NDBI'),
        'MNDWI': processed_2025.select('MNDWI'),
        'SAVI':  processed_2025.select('SAVI')
    }
).rename('BuiltUp_2025')

# ── 2016 Built-up Mask ───────────────────────────────────────
built_up_2016 = processed_2016.expression(
    '(NDBI > 0.13) && (MNDWI < 0) && (SAVI < 0.18)',
    {
        'NDBI':  processed_2016.select('NDBI'),
        'MNDWI': processed_2016.select('MNDWI'),
        'SAVI':  processed_2016.select('SAVI')
    }
).rename('BuiltUp_2016')

print("Built-up masks ready: 2016 and 2025.")

---
### Land Transformation Map

A two-panel view of:
- **Built-up change** (2016 → 2025) — reusing the 4-class growth heatmap from RQ1
- **Vegetation loss** — NDVI/SAVI change between 2016 and 2025

Both layers are shown together so you can read **where built-up growth co-occurred with vegetation loss**.

Built-up Growth Heatmap (2016 → 2025)

| Value | Class | Colour |
|---|---|---|
| 0 | No built-up (both years) | ⬛ |
| 1 | Stable built-up | ⬜ |
| 2 | New growth 2016→2025 | 🟠 |
| 3 | Lost built-up | 🔵 |

In [ ]:
# ── Built-up Change Classification ──────────────────────────
change_stack = built_up_2016.rename('y2016').addBands(built_up_2025.rename('y2025'))

growth_class = change_stack.expression(
    "(b('y2016') == 0 && b('y2025') == 0) ? 0"   # No built-up
    ": (b('y2016') == 1 && b('y2025') == 1) ? 1"  # Stable
    ": (b('y2016') == 0 && b('y2025') == 1) ? 2"  # New Growth
    ": (b('y2016') == 1 && b('y2025') == 0) ? 3"  # Lost
    ": 0"
).rename('GrowthClass').clip(roi)

### Vegetation Loss Layer (NDVI change & SAVI change)

Negative delta = vegetation lost.  
We compute both NDVI delta and SAVI delta - SAVI is more reliable over Dholera's arid/saline background.

In [ ]:
# ── Vegetation Change Layers ─────────────────────────────────

# SAVI delta: more soil-noise-robust over arid landscapes
savi_2025 = processed_2025.select('SAVI')
savi_2016 = processed_2016.select('SAVI')
savi_delta = savi_2025.subtract(savi_2016).rename('SAVI_Delta').clip(roi)

print("Vegetation change layers computed.")

#### Visualising SAVI and change in SAVI (2025-2016)

In [ ]:

# --- Map 3: SAVI (Testing)---
Map_Test = geemap.Map()
Map_Test.centerObject(roi, 10)
test_params = {'min': 0, 'max': 1, 'palette': ['8b0000', 'f5f5f5', '006400']}

Map_Test.addLayer(savi_2025, test_params, 'SAVI (2025)')
Map_Test.addLayer(savi_2016, test_params, 'SAVI (2016)')
Map_Test.add_colorbar(vis_params=test_params, label="Test Value", orientation="horizontal")
print("Displaying Test Map...")
display(Map_Test)

In [ ]:
# --- Delta Calculations ---
Map_Test = geemap.Map()
Map_Test.centerObject(roi, 10)
# SAVI Delta: (Later Year - Earlier Year)
# Positive values = Increase in vegetation/soil health
# Negative values = Decrease/Degradation
savi_delta = savi_2025.subtract(savi_2016)

# Define a specific Delta Palette (Red for decrease, White for neutral, Blue/Green for increase)
delta_params = {'min': -0.5, 'max': 0.5, 'palette': ['8b0000', 'f5f5f5', '006400']}

# Add to a new map or the existing one
Map_Test.addLayer(savi_delta, delta_params, 'SAVI Change (2016-2025)')

Map_Test.add_colorbar(vis_params=delta_params, label="Delta Value", orientation="horizontal")

print("Delta layers calculated and added.")
display(Map_Test)

It is observed that net SAVI of 2025 is actually higher in the whole region of Dholera, because of higher monsoon that year in 2025 compared to 2016 in the same time period of (Oct-Dec). Hence the vegetation cover of 2025 in total is actually higher. 
 
That's why we specifically choose to check change in Savi in the region of active construction. This will give us an accurate picture of how much vegetation is lost due to vegetation.

#### OUTPUT 1 : Visualise: Land Transformation Map

In [ ]:
# ── Output 1: Built-up Transformation Map ────────────────────
# Logic: Growth Class == 2 (New Growth) AND SAVI Delta < 0
# This isolates pixels that urbanised AND lost vegetation simultaneously

# Step 1: Mask SAVI delta to new growth footprint only
new_growth_mask   = growth_class.eq(2)                        # Class 2 = new growth pixels
veg_loss_in_growth = savi_delta.updateMask(                   # Keep only pixels where:
    new_growth_mask                                            #   (a) new growth occurred
    .And(savi_delta.lt(0))                                     #   (b) SAVI declined
)

# Step 2: New growth footprint as a flat base layer (charcoal)
new_growth_flat = new_growth_mask.selfMask()

# ── Map ──────────────────────────────────────────────────────
Map_LT = geemap.Map()
Map_LT.centerObject(roi, 11)

# Reference
Map_LT.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# Layer 1: Full new-growth footprint as flat charcoal base
# This shows the full 34 km² even where SAVI didn't decline
Map_LT.addLayer(
    new_growth_flat,
    {'min': 1, 'max': 1, 'palette': ['444444']},
    '⬛ New Growth Footprint (34 km²)'
)

# Layer 2: Vegetation loss gradient — overlaid WITHIN the new growth footprint
# Palette: pale yellow (marginal loss) → deep red (severe loss)
# min is set to -0.3 instead of -0.5 to stretch the colour ramp
# across the realistic SAVI loss range for Dholera's arid landscape
Map_LT.addLayer(
    veg_loss_in_growth,
   # {'min': -0.3, 'max': 0.0, 'palette': ['ffffb2', 'fd8d3c', 'e31a1c', '800026']},
    {'min': -0.3, 'max': 0.0, 'palette': ['800026','e31a1c','fd8d3c','ffffb2']},
    '🔴 Vegetation Loss Intensity (SAVI Decline in New Growth)'
)

# Colorbar
Map_LT.add_colorbar(
    vis_params={'min': -0.3, 'max': 0.0,
                'palette': ['800026','e31a1c','fd8d3c','ffffb2']},
    label='ΔSAVI (0 = No Loss  →  −0.3 = Severe Loss)',
    orientation='horizontal'
)

Map_LT.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Output 1: Built-up Transformation Map ready.")
display(Map_LT)

### Vegetation Loss Area Statistics

Measures the area where SAVI *declined* (vegetation lost) within the new growth footprint - directly tests **H1**.

In [ ]:
# ── Step 3: Key Statistic — Average SAVI Loss per km² of New Growth ──
# Mean SAVI delta across the masked (new growth AND veg loss) zone
mean_savi_loss = savi_delta.updateMask(new_growth_mask.And(savi_delta.lte(0))) \
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    ).getInfo()

pixel_area = ee.Image.pixelArea()

# Area of new growth with confirmed veg loss
veg_loss_area_m2 = new_growth_mask.And(savi_delta.lt(0)) \
    .multiply(pixel_area) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=10,
        maxPixels=1e13
    ).getInfo()

avg_savi_loss   = round(mean_savi_loss.get('SAVI', 0), 4)
veg_loss_km2    = round(list(veg_loss_area_m2.values())[0] / 1e6, 3)
total_growth_km2 = 34.013   # from RQ1

print()
print("=" * 55)
print("  OUTPUT 1 — KEY STATISTICS")
print("=" * 55)
print(f"  🟠 New Built-up Growth (2016→2025)     : {total_growth_km2} km²")
print(f"  🌿 Growth ∩ Vegetation Loss Area        : {veg_loss_km2} km²")
if total_growth_km2 > 0:
    pct = round(veg_loss_km2 / total_growth_km2 * 100, 1)
    print(f"  → {pct}% of new growth occurred at cost of local biomass")
print(f"  📉 Avg SAVI Loss per km² of New Growth  : {avg_savi_loss}")
print()
print("  Ecological Interpretation:")
print("  Each km² of new industrial footprint carries an average SAVI")
print(f"  decline of {avg_savi_loss} - the ecological price tag of corridor expansion.")
print("=" * 55)

---
## OUTPUT 2 : Heat Susceptibility Map

> **Proxy for urban heat stress: high built-up density + low vegetation**

This is a **composite index** - not a direct LST measurement - built from:
- `NDBI` (built-up surface): higher = more heat-absorbing materials
- `SAVI` (vegetation): higher = more evapotranspiration, lower heat stress

**Formula:**
```
Heat_Index = NDBI_norm - SAVI_norm
```
Values range ~[-1 to +1].  
+ Positive = high heat susceptibility (built-up, no vegetation)  
- Negative = low heat susceptibility (vegetated, no built-up)

In [ ]:
# ── Heat Susceptibility Index (2025) ─────────────────────────
# Normalise NDBI and SAVI to 0–1 range before differencing
# Using min–max normalisation anchored to realistic Sentinel-2 ranges

ndbi_raw  = processed_2025.select('NDBI')
savi_raw  = processed_2025.select('SAVI')

# NDBI normalised: range −0.5 to +0.5
ndbi_norm = ndbi_raw.add(0.5).divide(1.0).clamp(0, 1).rename('NDBI_norm')

# SAVI normalised: range 0 to 0.8
savi_norm = savi_raw.divide(0.8).clamp(0, 1).rename('SAVI_norm')

# Heat Susceptibility = Built-up signal minus Vegetation buffer
heat_index = ndbi_norm.subtract(savi_norm).rename('HeatIndex').clip(roi)

print("Heat Susceptibility Index computed.")

In [ ]:
# ── Visualise: Heat Susceptibility Map ───────────────────────
Map_Heat = geemap.Map()
Map_Heat.centerObject(roi, 11)

# Reference
Map_Heat.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# --- Layer 1: Continuous Heat Index (gradient) ---
# Palette: cool blue → white → fiery red
Map_Heat.addLayer(
    heat_index,
    {'min': -0.5, 'max': 0.7, 'palette': ['313695', '74add1', 'ffffbf', 'f46d43', 'a50026']},
    '🌡️ Heat Susceptibility Index (Continuous)'
)

# Colorbar for heat index
Map_Heat.add_colorbar(
    vis_params={'min': -0.5, 'max': 0.7, 'palette': ['313695', '74add1', 'ffffbf', 'f46d43', 'a50026']},
    label='Heat Susceptibility Index  \n (−0.5 = Cool/Vegetated → \n +0.7 = Hot/Built-up)',
    orientation='horizontal'
)

Map_Heat.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Heat Susceptibility Map ready.")
display(Map_Heat)

---
## OUTPUT 3 : Flood Risk Overlay

### Bivariate Flood Vulnerability Index (FVI)

**Approach:** Multi-criteria classification combining:
- **SAR-derived Inundation Frequency** (Sentinel-1, monsoon 2025) - radar penetrates cloud cover, directly measures water pooling
- **Terrain Analysis** (Copernicus GLO-30) - elevation

| Class | Criteria | Interpretation |
|---|---|---|
| 🚨 High (3) | `Elevation ≤ 5m` AND `Freq > 0.35` | Structural sink — floods every monsoon |
| ⚠️ Moderate (2) | `Elevation ≤ 8m` AND `Freq > 0.1` | Conditionally flood-prone |
| ✅ Low (1) | Everything else | Topographically resilient or no radar water signature |

---
### Stage 1: Terrain Preparation

Three terrain layers derived from Copernicus GLO-30:
- **Elevation** - absolute height above sea level
- I intentionally dropped in the idea of HAND for a coastal plane like Dholera, which is very flat, maximum slope was calculated at 0.038.
- As, its near sea level and not like Delhi or Mountain region, therefore adding HAND would not be a good idea.

In [ ]:
# ── Stage 1a: Load GLO-30 DEM ────────────────────────────────
# Reuse dem if already loaded earlier in notebook,
# otherwise load fresh here
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30') \
    .select('DEM') \
    .filterBounds(roi) \
    .median() \
    .clip(roi) \
    .rename('Elevation')

print("GLO-30 DEM loaded.")

In [ ]:
# ── Stage 1b: Permanent Water / Salt Pan Exclusion Mask ──────
# Source: Dry-season Sentinel-2 (Oct–Dec 2025)
# Pixels where MNDWI > 0.3 in the dry season = permanent water
# or salt pan — NOT monsoon flood risk
# These are excluded from FVI classification entirely

mndwi_dry = processed_2025.select('MNDWI')
permanent_water_mask = mndwi_dry.gt(0.2).rename('PermanentWater')

print("Permanent water / salt pan exclusion mask ready.")

In [ ]:
# ── Stage 1e: Visualise Terrain Inputs ───────────────────────
Map_Terrain = geemap.Map()
Map_Terrain.centerObject(roi, 11)

# Elevation
Map_Terrain.addLayer(
    dem,
    {'min': 0, 'max': 15,
     'palette': ['0d0887', '46039f', '7201a8', '9c179e',
                 'bd3786', 'd8576b', 'ed7953', 'fb9f3a', 'fdca26', 'f0f921']},
    '🏔️ GLO-30 Elevation (m)'
)


# Permanent water exclusion
Map_Terrain.addLayer(
    permanent_water_mask.selfMask(),
    {'min': 1, 'max': 1, 'palette': ['00b4d8']},
    '💧 Permanent Water / Salt Pan (Excluded from FVI)', shown=False
)

Map_Terrain.add_colorbar(
    vis_params={'min': 0, 'max': 15,
                'palette': ['0d0887', '46039f', '7201a8', '9c179e',
                            'bd3786', 'd8576b', 'ed7953', 'fb9f3a', 'fdca26', 'f0f921']},
    label='Elevation (m)',
    orientation='horizontal'
)

Map_Terrain.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("Terrain inputs ready — inspect before SAR processing.")
display(Map_Terrain)

---
### Stage 2 - SAR Inundation Frequency (Sentinel-1)

**Why SAR:** Sentinel-2 MNDWI is unusable during peak monsoon - cloud cover over Dholera in June-September routinely exceeds 50%. Sentinel-1 SAR (C-band) penetrates cloud cover completely. Open water is specularly smooth, returning near-zero backscatter - the contrast with land is unambiguous.

**Pipeline per image:**
1. Load - filter IW mode, VH band, descending orbit
2. Speckle filter (focal mean, 3×3) - must run per image before thresholding
3. Threshold at −20 dB → binary water mask
4. Subtract permanent water (salt pan exclusion)

**Output:** `.mean()` across collection = inundation frequency (0-1)

In [ ]:
# ── Stage 2a: Load and Filter Sentinel-1 ─────────────────────
# IW (Interferometric Wide) mode — standard for land applications
# VH polarisation — cross-pol; better water/land contrast in flat terrain
# Descending orbit — more consistent geometry over Gujarat
# Cloud threshold not applied — SAR is cloud-independent

s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(roi) \
    .filterDate('2025-06-15', '2025-09-30') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .select('VH')

print(f"Sentinel-1 collection size: ", s1_collection.size().getInfo(), "images")

In [ ]:
# 1. Create a median composite to clear up some "speckle" (noise)
s1_median = s1_collection.median().clip(roi)

# 2. Define visualization parameters
# VH values usually range from -25 (low backscatter/smooth surfaces like water) 
# to 0 (high backscatter/rough surfaces or vegetation)
s1_vis = {
    'min': -25,
    'max': 0, 
    'palette': ['black', 'white'] # This creates the greyish radar look
}

# 3. Add to the map
Map.centerObject(roi, 10)
Map.addLayer(s1_median, s1_vis, 'Sentinel-1 VH (Grayscale)')
Map

In [ ]:
# ── Stage 2b: Per-image Processing Pipeline ──────────────────
# Applied as a mapped function so each image is processed
# independently before compositing

def process_sar_image(image):
    """
    Per-image SAR flood detection pipeline:
    1. Speckle filter: focal mean 3x3 kernel
       Must run BEFORE thresholding — filtering a binary image
       destroys the water/land signal
    2. Threshold: VH < -20 dB = water (1), else land (0)
    3. Exclude permanent water / salt pans
    """
    # Step 1: Speckle filter
    smoothed = image.focal_mean(
        radius=1, kernelType='square', units='pixels'
    )

    # Step 2: Binary water mask
    # -20 dB threshold: standard for open water in flat, low-veg terrain
    water_binary = smoothed.lt(-20).rename('Water')

    # Step 3: Remove permanent water / salt pans
    # Subtracting the dry-season permanent water mask isolates
    # monsoon inundation from background water features
    monsoon_water = water_binary.where(permanent_water_mask.eq(1), 0)

    return monsoon_water.clip(roi) \
        .copyProperties(image, ['system:time_start'])


s1_processed = s1_collection.map(process_sar_image)

print("Per-image SAR pipeline applied.")

In [ ]:
# ── Stage 2c: Inundation Frequency Surface ───────────────────
# .mean() across the binary collection
# Result: continuous 0–1 raster
#   1.0 = flooded in every SAR pass
#   0.1 = flooded in ~10% of passes (flash inundation)
#   0.0 = never showed water signature

flood_freq = s1_processed.mean().rename('FloodFreq').clip(roi)

print("Inundation frequency surface computed.")
print()
print("Frequency interpretation:")
print("  > 0.60  — Semi-permanent inundation (natural depression)")
print("  0.35–0.60 — Recurring monsoon flooding (structural sink)")
print("  0.10–0.35 — Episodic flooding (peak rain events)")
print("  0.05–0.10 — Flash inundation (drains within days)")
print("  < 0.05  — Effectively non-flooded")

In [ ]:
# ── Stage 2d: Visualise SAR Flood Frequency ──────────────────
Map_SAR = geemap.Map()
Map_SAR.centerObject(roi, 11)

# Reference: True colour (dry season)
Map_SAR.addLayer(
    s2_2025, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'S2 True Colour (2025)', shown=False
)

# Flood frequency gradient
# Palette: white (never flooded) → teal → deep blue (always flooded)
# ['f7fbff', 'c6dbef', '6baed6', '2171b5', '08306b']
Map_SAR.addLayer(
    flood_freq,
    {'min': 0, 'max': 1, 'palette': ['08306b','2171b5','6baed6','c6dbef','f7fbff' ]},
    '🌊 SAR Inundation Frequency (Jun–Sep 2025)'
)

# High-frequency zones only (> 0.35) — these become High Risk
Map_SAR.addLayer(
    flood_freq.updateMask(flood_freq.gt(0.35)),
    {'min': 0.35, 'max': 1.0, 'palette': ['f4a261', 'd62828']},
    '🚨 High Frequency Zones (> 0.35) — candidate High Risk', shown=False
)

Map_SAR.add_colorbar(
    vis_params={'min': 0, 'max': 1,
                'palette': ['f7fbff', 'c6dbef', '6baed6', '2171b5', '08306b']},
    label='Inundation Frequency  (0 = Never flooded → 1 = Flooded every pass)',
    orientation='horizontal'
)

Map_SAR.addLayer(
    roi.style(color='00ffcc', fillColor='00000000', width=2), {},
    'Dholera Boundary'
)

print("SAR Inundation Frequency map ready.")
display(Map_SAR)